# React Flow in Solveit

Run this dialog from top to bottom, or use **Run all** to create, validate, launch, and display a React Flow development app without opening a separate terminal.

The workflow uses Solveit’s mapped port `8000`, runs Vite in a persistent fastmux session, and embeds the live app below. Saving any file under `my-react-flow-app/src/` sends an automatic full-reload message over Vite’s public WebSocket, refreshing the iframe and any open app tab without rerunning a cell. A `.solveit-react-flow.json` state file prevents later **Run all** executions from overwriting source files after initialization succeeds.

If at any time you need to reset the app, you can just ask the LLM to reset back to the default React Flow app, or manually delete the `my-react-flow-app/` project folder and `.solveit-react-flow.json`set up file (and then re run all code cells).

## 1. Configure the workflow

This cell establishes the app directory, mapped URL, fastmux session name, and persistent setup state.

On the first run, `INITIALIZE` is true. Once the starter app builds successfully, the state file records `initialized: true`. Future **Run all** executions preserve the existing app and its source edits. If a first-time setup is interrupted, rerunning safely deletes only the incomplete app created under the still-uninitialized state and starts it again.


In [ ]:
# Shared configuration and persistent initialization state
from pathlib import Path
import importlib, importlib.util, json, os, shutil, subprocess, sys

APP = Path("my-react-flow-app")
CONFIG = Path(".solveit-react-flow.json")
PORT = 8000
SID = "react-flow-dev"
CONFIG_SCHEMA = 1

DOMAINS = json.loads(os.environ["PUBLIC_DOMAINS"])
if str(PORT) not in DOMAINS:
    raise RuntimeError(f"Port {PORT} has no PUBLIC_DOMAINS mapping")
PUBLIC_HOST = DOMAINS[str(PORT)]
PUBLIC_URL = f"https://{PUBLIC_HOST}"

if CONFIG.exists():
    try:
        STATE = json.loads(CONFIG.read_text())
    except Exception as exc:
        raise RuntimeError(f"Invalid JSON in {CONFIG}") from exc
    if STATE.get("schema") != CONFIG_SCHEMA:
        raise RuntimeError(f"Unsupported config schema in {CONFIG}")
else:
    if APP.exists():
        raise RuntimeError(
            f"{APP} exists without {CONFIG}; refusing to overwrite an untracked app. "
            "Rename or remove the directory explicitly before rerunning."
        )
    STATE = {
        "schema": CONFIG_SCHEMA,
        "initialized": False,
        "port": PORT,
        "session": SID,
    }
    CONFIG.write_text(json.dumps(STATE, indent=2) + "\n")

INITIALIZE = STATE.get("initialized") is not True

if INITIALIZE and APP.exists():
    shutil.rmtree(APP)
    print(f"Removed incomplete setup: {APP}")

if importlib.util.find_spec("fastmux") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "fastmux"], check=True)
    importlib.invalidate_caches()

print(f"Mode: {'initialize starter app' if INITIALIZE else 'preserve existing app'}")
print(f"App directory: {APP.resolve()}")
print(f"Public URL: {PUBLIC_URL}")

Mode: preserve existing app


App directory: /app/data/Tests/Node/React Flow/MLP/my-react-flow-app


Public URL: https://nice-pond-arrives-x8yg3d.solveit.pub


## 2. Create the project and install dependencies

On the first run this cell scaffolds a Vite React project and installs `@xyflow/react`. Later runs skip project creation. If `node_modules` has been removed from an initialized project, the cell restores dependencies from the lock file.


In [ ]:
# Create the Vite project once; restore missing dependencies when needed
if INITIALIZE:
    subprocess.run(
        ["npm", "create", "vite@latest", APP.name, "--", "--template", "react"],
        check=True,
    )
    subprocess.run(["npm", "install", "@xyflow/react"], cwd=APP, check=True)
elif not (APP / "node_modules").exists():
    command = ["npm", "ci"] if (APP / "package-lock.json").exists() else ["npm", "install"]
    subprocess.run(command, cwd=APP, check=True)
else:
    print("Project and dependencies already exist; preserved them.")

Project and dependencies already exist; preserved them.


## 3. Write the starter application once

These cells write `vite.config.js`, `src/App.jsx`, and `src/index.css` only while `INITIALIZE` is true. After the first successful build sets the initialization flag, future **Run all** executions skip these writes and preserve source changes.

After initialization, edit files directly under `my-react-flow-app/src/` using Solveit’s file editor. Every saved source change automatically reloads only the app document through Vite, so tutorial code can remain conventional, no special React state-reset structure is required. Update these starter snippets only when intentionally creating a new baseline.

In [ ]:
# Write the Solveit-aware Vite configuration only during initialization
from textwrap import dedent

if INITIALIZE:
    vite_config = dedent("""\
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

const port = Number(process.env.SOLVEIT_PORT || 8000)
const domains = JSON.parse(process.env.PUBLIC_DOMAINS || '{}')
const publicHost = domains[String(port)]

const tutorialReload = {
  name: 'solveit-tutorial-reload',
  handleHotUpdate({ file, server }) {
    if (!file.includes('/src/')) return
    server.ws.send({ type: 'full-reload' })
    return []
  },
}

export default defineConfig({
  plugins: [react(), tutorialReload],
  server: {
    host: '0.0.0.0',
    port,
    strictPort: true,
    ...(publicHost ? { allowedHosts: [publicHost] } : {}),
    ...(publicHost ? { ws: { protocol: 'wss', host: publicHost, clientPort: 443 } } : {}),
  },
})
""")
    (APP / "vite.config.js").write_text(vite_config)
    print("Wrote vite.config.js")
else:
    print("Preserved vite.config.js")

Preserved vite.config.js


In [ ]:
# Write the React Flow starter component only during initialization
from textwrap import dedent

if INITIALIZE:
    app_source = dedent("""\
import { useCallback } from 'react'
import {
  Background,
  BackgroundVariant,
  Controls,
  MiniMap,
  ReactFlow,
  addEdge,
  useEdgesState,
  useNodesState,
} from '@xyflow/react'

const initialNodes = [
  {
    id: 'n1',
    type: 'input',
    position: { x: 0, y: 0 },
    data: { label: 'Node 1' },
  },
  {
    id: 'n2',
    type: 'output',
    position: { x: 0, y: 120 },
    data: { label: 'Node 2' },
  },
]

const initialEdges = [
  { id: 'n1-n2', source: 'n1', target: 'n2' },
]

export default function App() {
  const [nodes, , onNodesChange] = useNodesState(initialNodes)
  const [edges, setEdges, onEdgesChange] = useEdgesState(initialEdges)

  const onConnect = useCallback(
    (connection) => setEdges((current) => addEdge(connection, current)),
    [setEdges],
  )

  return (
    <ReactFlow
      nodes={nodes}
      edges={edges}
      onNodesChange={onNodesChange}
      onEdgesChange={onEdgesChange}
      onConnect={onConnect}
      fitView
      fitViewOptions={{ padding: 0.2 }}
      colorMode="system"
    >
      <Controls />
      <MiniMap pannable zoomable />
      <Background variant={BackgroundVariant.Dots} gap={12} size={1} />
    </ReactFlow>
  )
}
""")
    (APP / "src" / "App.jsx").write_text(app_source)
    print("Wrote src/App.jsx")
else:
    print("Preserved src/App.jsx")

Preserved src/App.jsx


In [ ]:
# Write the global stylesheet only during initialization
from textwrap import dedent

if INITIALIZE:
    index_css = dedent("""\
@import '@xyflow/react/dist/style.css';

html,
body,
#root {
  width: 100%;
  height: 100%;
  margin: 0;
}

* {
  box-sizing: border-box;
}
""")
    (APP / "src" / "index.css").write_text(index_css)
    print("Wrote src/index.css")
else:
    print("Preserved src/index.css")

Preserved src/index.css


## 4. Validate and commit initialization

This cell runs a production build every time. On the first successful build it atomically changes `.solveit-react-flow.json` to `initialized: true`. The flag is never set after a failed scaffold, dependency install, source write, or build.


In [ ]:
# Validate required starter sources, build, and commit first-time initialization
from textwrap import dedent

if INITIALIZE:
    required_css = dedent("""\
@import '@xyflow/react/dist/style.css';

html,
body,
#root {
  width: 100%;
  height: 100%;
  margin: 0;
}

* {
  box-sizing: border-box;
}
""")
    css_path = APP / "src" / "index.css"
    if not css_path.exists() or css_path.read_text() != required_css:
        css_path.write_text(required_css)
        print("Installed required src/index.css before validation")

if INITIALIZE:
    required_markers = {
        APP / "src" / "App.jsx": ("<ReactFlow",),
        APP / "src" / "index.css": ("@xyflow/react/dist/style.css",),
        APP / "vite.config.js": (
            "ws: { protocol: 'wss'",
            "solveit-tutorial-reload",
            "type: 'full-reload'",
        ),
    }
    missing = [str(path) for path, markers in required_markers.items()
               if not path.exists()
               or any(marker not in path.read_text() for marker in markers)]
    if missing:
        raise RuntimeError(f"Starter validation failed for: {', '.join(missing)}")
else:
    print("Skipped starter-source validation to preserve existing edits.")

subprocess.run(["npm", "run", "build"], cwd=APP, check=True)

if INITIALIZE:
    STATE["initialized"] = True
    STATE["port"] = PORT
    STATE["session"] = SID
    temp_config = CONFIG.with_suffix(CONFIG.suffix + ".tmp")
    temp_config.write_text(json.dumps(STATE, indent=2) + "\n")
    temp_config.replace(CONFIG)
    INITIALIZE = False
    print(f"Initialization committed in {CONFIG}")
else:
    print("Build passed; initialization state was already committed.")

Skipped starter-source validation to preserve existing edits.


Build passed; initialization state was already committed.


## 5. Start Vite with fastmux

This cell is safe during **Run all**. It closes the named fastmux session, terminates any remaining listener on port `8000`, starts Vite in a new persistent background session, and waits until the local server responds. This prevents stale-session and “port already in use” errors without requiring a separate terminal.

Saving a file under `my-react-flow-app/src/` makes Vite automatically reload the app document in the existing iframe and any open app tab. This does not rerun the cell or reload the Solveit dialog, but it does reset the app’s runtime state. Normal source edits do not require rerunning this cell.

In [ ]:
# Restart Vite in a persistent fastmux session on port 8000
import os, signal, subprocess, time
from urllib.request import urlopen
from fastmux.bg import close, display as mux_display, poll, start_session

try:
    close(SID)
    print(f"Closed fastmux session: {SID}")
except Exception:
    pass

result = subprocess.run(
    ["lsof", "-t", "-i", f":{PORT}"],
    capture_output=True,
    text=True,
)
pids = sorted({int(value) for value in result.stdout.split() if value.isdigit()})

for pid in pids:
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

if pids:
    time.sleep(0.5)
    remaining = subprocess.run(
        ["lsof", "-t", "-i", f":{PORT}"],
        capture_output=True,
        text=True,
    )
    for value in remaining.stdout.split():
        if value.isdigit():
            try:
                os.kill(int(value), signal.SIGKILL)
            except ProcessLookupError:
                pass
    print(f"Cleared port {PORT}: {pids}")
else:
    print(f"Port {PORT} was free")

start_session(
    SID,
    cmd=["npm", "run", "dev"],
    cwd=str(APP.resolve()),
    env={"SOLVEIT_PORT": str(PORT)},
    width=120,
    height=30,
)
poll(SID, wait_ms=3000)

local_url = f"http://127.0.0.1:{PORT}"
for _ in range(40):
    try:
        with urlopen(local_url, timeout=0.5) as response:
            if response.status < 500:
                break
    except Exception:
        time.sleep(0.25)
else:
    mux_display(SID, lines=40)
    raise RuntimeError(f"Vite did not become ready at {local_url}")

print(f"Vite is ready: {PUBLIC_URL}")
mux_display(SID, lines=30)

Closed fastmux session: react-flow-dev


Port 8000 was free


Vite is ready: https://nice-pond-arrives-x8yg3d.solveit.pub


```python

> my-react-flow-app@0.0.0 dev
> vite


  VITE v8.2.2  ready in 164 ms

  ➜  Local:   http://localhost:8000/
  ➜  Network: http://172.18.0.44:8000/  eth0
  ➜  press h + enter to show help
── react-flow-dev:0.0 %0 · lines 0-10 of 10
```

## 6. View the live application

The iframe is the live app, not a screenshot. Drag nodes, use the controls, or open the same mapped URL in another tab. Saving any file under `my-react-flow-app/src/` automatically reloads the app document in the iframe and open app tabs through Vite’s WebSocket. The Solveit dialog remains loaded and no cells rerun.

In [ ]:
# Display the live React Flow app inline
import time
from IPython.display import HTML, IFrame, display

iframe_url = f"{PUBLIC_URL}?solveit_run={time.time_ns()}"
display(HTML(
    f'<p><a href="{PUBLIC_URL}" target="_blank">Open React Flow in a new tab</a></p>'
))
display(IFrame(iframe_url, width="100%", height=700))

HTML(<p><a href="https://nice-pond-arrives-x8yg3d.solveit.pub" target="_blank">Open React Flow in a new tab</a></p>)

## 7. Develop with automatic source reload

After the first successful run:

1. Edit and save files under `my-react-flow-app/src/` in Solveit.
2. The existing iframe and any open app tab reload automatically through Vite’s WebSocket; do not rerun the iframe cell or refresh manually.
3. Expect runtime state such as dragged node positions, selections, and form input to reset after each source save.
4. Rerun the build cell when you want full production validation.
5. Rerun the fastmux cell only after changing dependencies, environment variables, port configuration, `vite.config.js`, or when the server has stopped.

You may safely use **Run all** again: the initialization flag skips project creation and starter-file writes while still rebuilding, restarting Vite cleanly on port `8000`, and recreating the iframe.

To inspect recent Vite output later:

```python
from fastmux.bg import display
display("react-flow-dev", lines=40)
```

Keep the duplicate backup dialog until this workflow has passed a clean first run and a second run after a test source edit.

### Quick automatic-reload test

Use a conventional tutorial edit. Add a third node to `initialNodes` in `src/App.jsx`:

```jsx
  {
    id: 'n3',
    type: 'output',
    position: { x: 220, y: 120 },
    data: { label: 'Source reload worked!' },
  },
```

Then add its edge to `initialEdges`:

```jsx
  { id: 'n2-n3', source: 'n1', target: 'n3' },
```

Right-click in the Solveit editor and select **Save File**. The iframe and any already-open app tab should reload automatically and show the third node, without rerunning any cell or manually refreshing. Edit the new label and save again to confirm repeated reloads.

After the test, keep the changes as the start of the tutorial or remove the third node and edge.

## 8. Edit the app with SolveitAI

After running the next cell—or using **Run all**—SolveitAI has the context and tools needed to inspect and edit the current React Flow source files.

For source-editing requests, SolveitAI must follow this workflow:

1. Treat the current project files as authoritative. Work inside `my-react-flow-app/src/` unless you explicitly request dependency, setup, or Vite changes.
2. Inspect all relevant source files before editing; inspect `package.json` when library versions or APIs matter. Preserve existing structure, imports, naming, and styling conventions.
3. Use `fd`/`rg` to discover files and symbols, `view_file` for read-only inspection, `lnhashview_file` immediately before editing, `file_exhash` for verified existing-file edits, and `create_file` for new files.
4. Make only the requested change. Apply related edits atomically and bottom-to-top; prefer `c` on complete short lines or blocks over fragile substring substitutions.
5. If an edit or hash check fails, stop, re-read, and retry the same agreed change—never guess, broaden scope, or leave a partial edit.
6. Report the resulting diff. Use the existing build cell only when full production validation is requested.

Every saved file under `src/` automatically reloads the app document in the iframe and open app tabs. The Solveit dialog and kernel remain running, but app runtime state resets. Tutorial source stays conventional React code.

In [ ]:
# Import the functions exposed as SolveitAI tools in the next note
from rgapi.skill import fd, rg
from fastcore.tools import create_file, view_file
from exhash.skill import file_exhash, lnhashview_file

def react_flow_exhash(
    path: str,
    cmds: list[tuple],
    sw: int = 4,
    inplace: bool = True,
):
    """Apply hash-verified edits to an existing React Flow source file."""
    source_root = (APP / "src").resolve()
    target = Path(path).resolve()
    if not target.is_relative_to(source_root):
        raise ValueError(f"Refusing to edit outside {source_root}: {target}")
    return file_exhash(target, *(tuple(cmd) for cmd in cmds), sw=sw, inplace=inplace)

print("React Flow source tools ready.")

React Flow source tools ready.


Use these tools for automated React Flow source work:

- &`fd` — discover files under the project.
- &`rg` — search source content and symbols.
- &`view_file` — inspect a file when no edit is planned.
- &`lnhashview_file` — obtain fresh hash-verified line addresses immediately before editing.
- &`react_flow_exhash` — the project-specific write tool. It adapts the exposed `cmds=[...]` tool schema to exhash's positional command API, restricts writes to `my-react-flow-app/src/`, applies related hash-verified edits atomically, and returns the diff.

`react_flow_exhash` is needed because exposing variadic `file_exhash(path, *cmds)` directly causes the tool adapter to pass an unsupported `cmds=` keyword. Once the code cell above has run, this wrapper allows SolveitAI to make automated source edits directly instead of generating a code cell for manual execution.

For speed, inspect only the relevant files, obtain one fresh `lnhashview_file` immediately before editing, then make the agreed related changes in one bottom-to-top `react_flow_exhash` call. Do not repeatedly rediscover the project, reread unchanged files, or broaden the task unless an edit fails.
- &`create_file` — create new component, module, or stylesheet files without overwriting existing files by default.

### Example request

Once the source tools are ready, you can ask for changes in ordinary language:

```text
Add node `n4`, connect `n1` to `n4`, and style the new node.
```

SolveitAI will inspect the current project files, make only the requested edits, report the diff, and rely on the existing automatic source reload to update the iframe and any open app tab.

### Automated React Flow  Teacher

You can also ask SolveIt to teach you React Flow. Ask it to explain concepts, show code, and automatically apply the code changes in realtime! Try out this prompt:

```text
Ok lets do a basic React Flow tutorial. Teach me the basic concepts one at a time, in bite-size steps. Restore the app to the default, and then show me the code changes, and apply them to the live running app.
```

### Try Something Random

Or jut ask the LLM to show you something cool about React Flow. This works pretty well and can come up with some interesting examples:

```text
Show me something cool you can do with React Flow. try to impress me!
```

## 9. Interactive MLP Laboratory

The default React Flow implementation was replaced with a mini interactive multilayer perceptron (MLP) laboratory. You can use it to create neural-network architectures, configure layers and activations, inspect weights, predictions, loss, and gradients, and run model-training steps directly in the interactive graph. Full details in the next section.

## 10. Understanding and experimenting with the included models

The app includes four deterministic synthetic examples:

1. **Mars lander survival** — binary classification and the default example.
2. **Residential sale-price estimation** — nonlinear regression.
3. **Alien-planet biome classification** — five-class classification.
4. **Robot grasp success** — binary classification with a safe operating range.

The examples are designed for CPU training and repeatable experiments. Their realism does not come from correlation alone: they combine correlated measurements, nonlinear interactions, context-dependent thresholds, probabilistic or noisy targets, hidden influences, and larger validation sets. Consequently, training should reveal useful signal without making perfect validation performance the expected outcome.

### Reading the learning-curve chart

Each training run now records metrics **before training** and again **after every epoch**. The chart appears while training is running and remains available with the completed run results.

The chart shows:

- **Training loss**: error on examples that contribute gradients.
- **Validation loss**: error on unseen examples that never contribute gradients.
- **Validation accuracy**: an optional classification-only curve, controlled by the **Validation accuracy** checkbox.

Click a point on the chart—or focus the chart and use the left and right arrow keys—to inspect a particular epoch. The metric cards below the chart report that epoch's exact training and validation loss, together with accuracy for classification or MAE for regression. Epoch **Start** is the baseline before the first parameter update.

A healthy run usually reduces both training and validation loss. A widening gap, where training loss continues falling while validation loss levels off or rises, is evidence of overfitting. The best model may therefore occur before the final epoch. Accuracy can also remain unchanged while cross-entropy improves, because the model may be assigning better probabilities without changing which class has the highest probability.

For useful comparisons, change one setting at a time and compare the whole curve—not only the final value. Keep the dataset split fixed when comparing learning rates, batch sizes, epoch counts, activations, or network capacity.

### Example 1: Mars lander survival

This default example is a **binary classification problem**. The network predicts whether a Mars lander completes a safe landing or the mission is lost.

#### What the data represents

The built-in dataset is deterministic, synthetic data for learning and experimentation. It is not mission data from a space agency. Resetting the example restores the same observations, making comparisons repeatable.

The default split contains:

- **480 training attempts**, used to calculate gradients and update parameters.
- **160 validation attempts**, used only to measure performance on unseen landings.

Every observed input is normalized to $[0,1]$:

| Feature | Interpretation |
|---|---|
| Descent speed | Larger values represent a faster final descent. |
| Fuel reserve | Remaining fuel available for powered descent and correction. |
| Payload mass | Relative landed mass. |
| Atmosphere density | Relative atmospheric density during descent. |
| Wind turbulence | Disturbance from variable winds. |
| Terrain slope | Difficulty of the selected landing surface. |
| Engine health | Reliability and available engine performance. |
| Navigation accuracy | Quality of position and trajectory estimates. |
| Parachute timing | Normalized deployment timing. Its best value depends on atmosphere density. |

The observations are not independent. Hidden mission-preparation and maintenance factors jointly influence fuel, navigation, and engine condition. Payload and atmosphere influence descent speed; weather affects both turbulence and terrain-selection difficulty.

The outcome also contains nonlinear effects. The safest parachute timing changes with atmosphere density, high speed is especially dangerous in thin atmosphere, heavy payload is worse when fuel is low, and turbulence matters more when navigation is poor.

The generator converts these effects into a latent safety score $s$ and samples the outcome using

$$
P(\text{safe landing})=\sigma(s)=\frac{1}{1+e^{-s}}.
$$

Random unobserved conditions are included before the outcome is sampled. Two attempts with similar visible measurements can therefore have different results. This creates overlapping classes and an irreducible error floor rather than a perfectly separable puzzle.

#### Network and objective

The default architecture is

$$
9\text{ inputs}\rightarrow12\text{ ReLU hidden units}\rightarrow2\text{ softmax outputs}.
$$

The outputs are probabilities for **safe landing** and **mission lost**, which sum to $1$. With one-hot target $y$, training minimizes cross-entropy:

$$
L=-\sum_k y_k\log(p_k).
$$

For each mini-batch of size $B$, the app averages fresh sample gradients and updates each parameter $\theta$ once:

$$
g_B=\frac{1}{B}\sum_{i=1}^{B}\nabla_\theta L_i,
\qquad
\theta\leftarrow\theta-\eta g_B.
$$

The default controls use **70 epochs**, mini-batches of **32**, and learning rate $\eta=0.05$.

#### What counts as success

Compare the whole learning curve rather than expecting near-perfect accuracy. A useful run should:

- reduce both training and validation cross-entropy;
- perform meaningfully above the 50% two-class baseline;
- keep the training and validation curves reasonably close;
- remain stable across later epochs.

Because outcomes are probabilistic, validation accuracy may fluctuate and should level off below 100%. Falling training loss with flat or rising validation loss is stronger evidence of overfitting than a noisy accuracy point by itself.

#### Knobs and experiments

- **Parachute timing:** Edit timing while holding atmosphere density fixed, then reverse the experiment. The best timing should depend on both features.
- **Interactions:** Compare a fast descent in dense and thin atmosphere, or a heavy payload with high and low fuel reserve.
- **Hidden units:** Try 4, 8, 12, and 20 units while keeping all training controls fixed.
- **Hidden layers:** Test whether a second layer helps represent the interaction structure. More depth may also make optimization harder.
- **Learning rate:** Compare values near `0.01`, `0.03`, `0.05`, and `0.08`; watch for slow progress or instability.
- **Training rows:** Reduce the dataset to observe noisier gradients and a larger train-validation gap.

Every new run restores the current model baseline and starts with fresh gradients. Manual topology or parameter changes establish the baseline for the next run; gradients never accumulate across runs.

### Example 2: Residential sale-price estimation

This example is a **regression problem**. The network predicts one continuous quantity: a residential property’s sale price.

#### What the data represents

The built-in dataset is deterministic, synthetic data designed to demonstrate realistic regression behaviour on a small CPU workload. It is not a real property-sales dataset and must not be used for valuation or financial decisions.

The default split contains:

- **480 training properties**, used to calculate gradients and update parameters.
- **160 validation properties**, held out from training to test predictions on unseen homes.

The seven observed inputs are normalized to $[0,1]$:

| Feature | Interpretation |
|---|---|
| Floor area | Relative property size. |
| Bedrooms | Relative bedroom capacity. |
| Bathrooms | Relative bathroom capacity. |
| Location score | Synthetic measure of neighbourhood desirability. |
| Property age | Larger values represent older homes. |
| Renovation quality | Larger values represent stronger renovation condition. |
| Transit access | Larger values represent stronger access to transport. |

The generator first samples hidden factors for neighbourhood prosperity, urban access, household scale, and upkeep investment. These shared causes create correlated observations: larger homes tend to have more bedrooms and bathrooms; prosperous, accessible areas tend to score better for location and transit; renovation quality depends partly on upkeep, neighbourhood, and property age.

The target then combines several nonlinear effects:

- floor area has diminishing returns through $\sqrt{f}$;
- extra floor area is worth more in a stronger location through $f\ell$;
- locations above a threshold receive an additional premium;
- renovation reduces the effective age penalty;
- bedroom value depends partly on whether the layout fits the home’s size.

A simplified description of two important terms is

$$
a_{\mathrm{effective}}=a(1-0.68q),
\qquad
p_{\mathrm{location}}=\max(0,\ell-0.62),
$$

where $a$ is age, $q$ is renovation quality, and $\ell$ is location score.

The target also includes unobserved amenities, noise that grows for unusual properties, and occasional unusual transactions. This means that even a well-trained network cannot reconstruct every sale exactly.

The app clips the generated target to $[0,1]$ and converts the network prediction to a displayed price using

$$
\text{displayed price}=\$1{,}200{,}000\times\hat y.
$$

For example, $\hat y=0.65$ is displayed as approximately **$780k**. The scale is part of this synthetic example, not a market valuation model.

#### Network and objective

The default architecture is

$$
7\text{ inputs}\rightarrow12\text{ ReLU hidden units}\rightarrow1\text{ linear output}.
$$

For one property, the app uses half squared error:

$$
L_i=\frac12(\hat y_i-y_i)^2.
$$

For a mini-batch of $B$ homes, it averages gradients and performs one update:

$$
g_B=\frac{1}{B}\sum_{i=1}^{B}\nabla_\theta L_i,
\qquad
\theta\leftarrow\theta-\eta g_B.
$$

The default controls use **80 epochs**, mini-batches of **32**, and learning rate $\eta=0.045$.

The result panel also reports mean absolute error:

$$
\operatorname{MAE}=\frac1N\sum_{i=1}^{N}|\hat y_i-y_i|.
$$

MAE is converted to the displayed dollar scale. A normalized MAE of $0.04$, for example, corresponds to approximately **$48k**.

#### What counts as success

A useful run should reduce validation loss and validation dollar MAE on the 160 unseen homes. Compare training and validation together:

- If both improve, the model is learning a relationship that generalizes.
- If training improves while validation stalls or worsens, the model may be overfitting.
- If neither improves, adjust optimization or capacity rather than merely adding epochs.
- If loss becomes unstable, reduce the learning rate.

Noise, hidden amenities, and unusual sales create an error floor. The goal is therefore a useful validation improvement—not memorization or zero error.

#### Knobs and experiments

- **Capacity:** Compare 4, 8, 12, and 20 hidden units with the same split and training controls.
- **Depth:** Add a second hidden layer to test whether it helps represent thresholds and interactions.
- **Learning rate:** Try values around `0.01`–`0.06` and compare complete curves.
- **Data quantity:** Reduce the training set while keeping the validation set fixed to observe generalization with limited evidence.
- **Nonlinearity:** Compare ReLU, sigmoid, and tanh. A purely linear model cannot represent all generator effects.
- **Feature interactions:** Inspect similarly sized homes in weak and strong locations, or old homes with low and high renovation quality.

A useful investigation is to record validation MAE for several capacities and prefer the smallest model that performs consistently well on unseen properties—not the model with the lowest training error.

### Example 3: Alien-planet biome classification

This example is a **five-class classification problem**. The network classifies an observed planet as **ocean**, **desert**, **ice**, **forest**, or **volcanic**.

#### What the data represents

The deterministic synthetic split contains **640 training planets** and **200 validation planets**. All nine observations are normalized to $[0,1]$: stellar radiation, atmospheric pressure, surface water, greenhouse strength, geological activity, gravity, axial tilt, mineral albedo, and magnetic shielding.

The observations share hidden causes and physical-style dependencies. Gravity influences pressure, geology, and shielding; pressure influences greenhouse strength and water retention; radiation, greenhouse strength, geology, and albedo jointly determine a hidden temperature; axial tilt is more disruptive when pressure is low.

Each biome receives a nonlinear suitability score. For example, forests favour water, pressure, shielding, and a temperate range; deserts favour warmth and low water; volcanic worlds favour geological activity; ice worlds favour low temperature. The generator samples a class from all five scores rather than selecting the largest score deterministically, so borderline planets can plausibly receive different labels.

#### Network and objective

The default architecture is

$$
9\text{ inputs}\rightarrow14\text{ ReLU hidden units}\rightarrow5\text{ softmax outputs}.
$$

Training minimizes multiclass cross-entropy over **80 epochs**, with mini-batches of **32** and learning rate $\eta=0.045$. A random five-class classifier has expected accuracy near 20%, but overlapping biomes and hidden variation make 100% validation accuracy unrealistic.

#### Experiments

- Compare validation cross-entropy and accuracy; probability quality can improve without changing the winning class.
- Inspect planets near the forest/ocean, desert/volcanic, and ice/ocean boundaries.
- Remove hidden units to demonstrate underfitting, then add capacity while watching for overfitting.
- Change one climate input at a time, then change a related input to reveal interactions.
- Compare one and two hidden layers with the same split and optimizer settings.

### Example 4: Robot grasp success

This example is a **binary classification problem**. The network predicts whether a robot achieves a secure grasp or drops or damages the object.

#### What the data represents

The deterministic synthetic split contains **480 training attempts** and **160 validation attempts**. The nine normalized inputs are object mass, object size, slipperiness, fragility, shape irregularity, gripper force, alignment error, approach angle, and sensor confidence.

The observations are correlated. Object size and hidden material density influence mass; density also affects slipperiness and fragility; irregular objects tend to produce poorer sensor confidence and larger alignment errors.

The central nonlinear effect is a **safe force window**. Too little force causes a drop, while too much force can crush a fragile object. The required force depends on mass, slipperiness, and shape; the crush limit depends on fragility and size. The best approach angle also changes with shape irregularity. Random unobserved variation is added before the success outcome is sampled probabilistically.

#### Network and objective

The default architecture is

$$
9\text{ inputs}\rightarrow12\text{ ReLU hidden units}\rightarrow2\text{ softmax outputs}.
$$

Training minimizes cross-entropy over **70 epochs**, with mini-batches of **32** and learning rate $\eta=0.055$. A useful model should beat the 50% two-class baseline while retaining a visible error floor caused by ambiguous and unobserved conditions.

#### Experiments

- Sweep gripper force from low to high for one object. Success should improve and then worsen rather than increasing forever.
- Compare the same force on heavy/slippery and light/fragile objects.
- Reduce sensor confidence and observe how uncertain alignment and force selection affect predictions.
- Compare 4, 8, 12, and 20 hidden units while watching both loss curves.
- Add a second hidden layer and test whether extra capacity improves validation results rather than only training loss.

## 11. Future work

Next potential tasks:

1. **Task-specific validation visualizations**
   - Mars and robot grasping: compact **confusion matrices** for the two outcomes.
   - Alien biomes: a five-class confusion matrix plus per-biome accuracy.
   - Housing: an **actual versus predicted** scatter plot and a list of the largest prediction errors.
   - These reveal which cases and classes the model handles poorly, rather than reducing performance to one score.

2. **Trivial baseline comparisons**
   - Binary classification: always predict the most common training class.
   - Alien biomes: always predict the most common biome.
   - Housing: always predict the mean training price.
   - Report the neural network and baseline side by side, including absolute and percentage improvement.
   - This answers the essential question: *Did the neural network learn anything useful?*

3. **Predictions in the validation table**
   - Show predicted class, class probabilities, and correctness for classification.
   - Show predicted price and absolute error for housing.
   - Highlight low-confidence classifications and unusually large regression errors.
   - Allow clicking a validation row to inspect its forward pass without training on it.

4. **Best-model checkpoint and early stopping**
   - Remember parameters from the epoch with the lowest validation loss.
   - At completion, report both the best and final epochs.
   - Provide **Use final model** and **Use best validation model** actions.
   - Optionally stop after validation loss fails to improve for a configurable number of epochs.

5. **Training progress and cancellation**
   - Compact status could show:
     ```text
     Epoch 18/70 · batch 7/15 · 26%
     ```
   - Add a thin progress bar and a clear **Stop training** action using the existing training state.

6. **Dataset controls**
   - Regenerate a dataset from a visible random seed.
   - Add difficulty presets such as **Clear signal**, **Realistic**, and **Noisy**.
   - Keep the current fixed datasets as reproducible defaults.
   - This demonstrates that extra model capacity cannot eliminate ambiguous labels, hidden influences, or noisy targets.

7. **Architecture summary and capacity warnings**
   - Display summaries such as:
     ```text
     9 → 12 → 2 · 146 trainable parameters
     ```
   - Update the summary whenever units or layers change.
   - Warn when parameter count is disproportionate to the number of training examples.

8. **Experiment presets**
   - **Underfit:** deliberately small network.
   - **Balanced:** the current recommended architecture.
   - **Overfit:** excessive capacity or a reduced training set.
   - Preserve the dataset split and random seed so comparisons remain meaningful.

9. **Human-readable input scales**
   - Keep normalized `0–1` values for explaining network mathematics, but add domain-specific captions or tooltips.
   - Examples:
     - housing floor area `0.72` → approximately `2,450 ft²`;
     - Mars descent speed `0.64` → a scenario-specific speed range;
     - gripper force `0.59` → a scenario-specific force range.
   - Clearly label these as synthetic scales rather than real engineering measurements.

10. **Experiment comparison history**
    - Record architecture, learning rate, batch size, epochs, best validation metric, and final validation metric for recent runs.
    - Let users compare runs while changing one variable at a time.
    - Make it easy to restore a promising architecture or parameter checkpoint.